# 02 – Partição, pré-processamento e baseline

Executa depois do notebook 01. Em modo demonstração, a tabela de métricas abaixo vem de dados sintéticos e **não deve ser reportada** como baseline do projeto.

In [1]:
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from src import config
from src.dados import carregar_bruto, limpar, agregar_mensal, validar_totais
from src.demo import gerar_registros_sinteticos, ROTULO_DEMO
pd.set_option("display.width", 140)
from src.variaveis import criar_variaveis
from src.particao import particionar_temporal
from src.preprocessamento import construir_preprocessador, resumo_do_fit
from src.baseline import avaliar_baselines

In [2]:
nome_real = config.PROCESSADO / "base_mensal.csv"
nome_demo = config.PROCESSADO / "base_mensal_DEMO.csv"
MODO_DEMO = not nome_real.exists()
arquivo = nome_demo if MODO_DEMO else nome_real
if MODO_DEMO:
    print(ROTULO_DEMO)
mensal = pd.read_csv(arquivo, parse_dates=[config.COL_MES])
print("Base lida:", arquivo.name, mensal.shape)

MODO DEMONSTRAÇÃO: dados sintéticos, não são dados do LAPAE.
Base lida: base_mensal_DEMO.csv (120, 4)


## Variáveis sem vazamento temporal
Defasagens e média móvel usam só meses anteriores ao mês-alvo.

In [3]:
feats, descartadas = criar_variaveis(mensal)
print("Linhas usadas:", len(feats), "| descartadas por falta de histórico ou alvo ausente:", descartadas)
feats.head()

Linhas usadas: 90 | descartadas por falta de histórico ou alvo ausente: 30


,unidade,mes,exames,municipio,lag_1,lag_2,lag_3,media_movel_3,mes_do_ano,dias_uteis
0,Unidade 01,2026-01-01,57,Município 1,42.0,61.0,41.0,48.000000,1,22
1,Unidade 01,2026-02-01,63,Município 1,57.0,42.0,61.0,53.333333,2,20
2,Unidade 01,2026-03-01,62,Município 1,63.0,57.0,42.0,54.000000,3,22
3,Unidade 01,2026-04-01,65,Município 1,62.0,63.0,57.0,60.666667,4,22
4,Unidade 01,2026-05-01,62,Município 1,65.0,62.0,63.0,63.333333,5,21


## Partição temporal
Treino no passado; validação e teste em meses posteriores. O teste fica guardado.

In [4]:
treino, valid, teste = particionar_temporal(feats)
for nome, d in [("treino", treino), ("validação", valid), ("teste", teste)]:
    print(f"{nome:10s} {d[config.COL_MES].min():%Y-%m} a {d[config.COL_MES].max():%Y-%m}  linhas={len(d)}")

treino     2026-01 a 2026-05  linhas=50
validação  2026-06 a 2026-07  linhas=20
teste      2026-08 a 2026-09  linhas=20


## Pré-processador ajustado só no treino

In [5]:
cols = config.NUMERICAS + config.CATEGORICAS
pre = construir_preprocessador().fit(treino[cols])
resumo = resumo_do_fit(pre)
print("Medianas aprendidas no treino:", resumo["medianas"])
print("Médias aprendidas no treino:", resumo["medias"])
print("Categorias (unidade):", resumo["categorias"]["unidade"][:5], "...")
formas = {n: pre.transform(d[cols]).shape for n, d in [("treino", treino), ("validação", valid), ("teste", teste)]}
print("Formas após transformar:", formas)
assert len({f[1] for f in formas.values()}) == 1, "Conjuntos com número de atributos diferente"

Medianas aprendidas no treino: [119.5, 115.5, 107.5, 114.83333333333333, 22.0]
Médias aprendidas no treino: [113.26, 109.64, 105.7, 109.53333333333335, 21.4]
Categorias (unidade): ['Unidade 01', 'Unidade 02', 'Unidade 03', 'Unidade 04', 'Unidade 05'] ...
Formas após transformar: {'treino': (50, 24), 'validação': (20, 24), 'teste': (20, 24)}


## Baselines no recorte de validação
Métricas: MAE, WAPE e viés (previsto − real).

In [6]:
tabela = avaliar_baselines(treino, valid).round(3)
tabela

,baseline,MAE,WAPE,viés
0,Persistência (operacional),11.30,0.085,-2.900
1,Média do treino,15.94,0.120,-15.100
2,Mediana do treino,15.40,0.116,-14.200
3,Média móvel de 3 meses,11.30,0.085,-8.833
